# 07 - Strict Time-Aware Split

Goal: evaluate the models under a stricter split than random sampling.

This notebook uses a class-aware time split: within each binary class, older traffic is used for training and newer traffic is used for validation/test. This keeps the binary evaluation balanced while still introducing temporal shift.

In [3]:
from argparse import Namespace
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.create_strict_splits import create_strict_splits  # noqa: E402
from src.build_graph_arrays import build_graph_arrays  # noqa: E402

STRICT_DIR = PROJECT_ROOT / "data" / "strict_time_balanced"
STRICT_RESULTS = PROJECT_ROOT / "results" / "strict_time_balanced"

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\Binh\OneDrive\Documents\CẦN NỘP\IDS_GAT_IOT23


## Create strict split

In [4]:
split_args = Namespace(
    input_path=PROJECT_ROOT / "data" / "processed" / "iot23_binary_sample.csv",
    output_dir=STRICT_DIR,
    strategy="class_time",
    rows_per_class=50_000,
    random_state=42,
)

split_metadata = create_strict_splits(split_args)
split_metadata["splits"]

{'train': {'rows': 70000,
  'binary_label_counts': {'0': 35000, '1': 35000},
  'raw_label_counts': {'Benign': 35000,
   'PartOfAHorizontalPortScan': 21212,
   'Okiru': 12293,
   'DDoS': 1321,
   'C&C': 160,
   'C&C-HeartBeat': 11,
   'C&C-FileDownload': 1,
   'C&C-Torii': 1,
   'Attack': 1},
  'ts_min': 1532101091.1943789,
  'ts_max': 1551381693.86339},
 'val': {'rows': 15000,
  'binary_label_counts': {'0': 7500, '1': 7500},
  'raw_label_counts': {'Benign': 7500,
   'PartOfAHorizontalPortScan': 7495,
   'Attack': 3,
   'C&C': 2},
  'ts_min': 1545408170.360159,
  'ts_max': 1552048400.8397882},
 'test': {'rows': 15000,
  'binary_label_counts': {'0': 7500, '1': 7500},
  'raw_label_counts': {'Benign': 7500,
   'DDoS': 4600,
   'PartOfAHorizontalPortScan': 2899,
   'C&C-HeartBeat': 1},
  'ts_min': 1547144827.497778,
  'ts_max': 1569018251.901591}}

## Build graph arrays

In [ ]:
graph_metadata = build_graph_arrays(
    processed_dir=STRICT_DIR,
    output_dir=STRICT_DIR,
)
import pandas as pd

train = pd.read_csv("data/strict_time_balanced/train.csv")
val = pd.read_csv("data/strict_time_balanced/val.csv")
test = pd.read_csv("data/strict_time_balanced/test.csv")

print(train.ts.min(), train.ts.max())
print(val.ts.min(), val.ts.max())
print(test.ts.min(), test.ts.max())

print(train.label.value_counts())

print(val.label.value_counts())

print(test.label.value_counts())
graph_metadata

Node feature dimension: 12
Average Packet Size sample:
[300.   0.   0.   0.   0.   0.   0.   0.   0.   0.]
train
label
Benign                       35000
PartOfAHorizontalPortScan    21212
Okiru                        12293
DDoS                          1321
C&C                            160
C&C-HeartBeat                   11
C&C-FileDownload                 1
C&C-Torii                        1
Attack                           1
Name: count, dtype: int64
val
label
Benign                       7500
PartOfAHorizontalPortScan    7495
Attack                          3
C&C                             2
Name: count, dtype: int64
test
label
Benign                       7500
DDoS                         4600
PartOfAHorizontalPortScan    2899
C&C-HeartBeat                   1
Name: count, dtype: int64


{'task': 'binary_edge_classification',
 'graph_design': {'node': 'IP address',
  'edge': 'traffic flow from id.orig_h to id.resp_h',
  'label': 'binary_label'},
 'model_feature_policy': {'numeric_columns': ['id.orig_p',
   'id.resp_p',
   'duration',
   'orig_bytes',
   'resp_bytes',
   'missed_bytes',
   'orig_pkts',
   'orig_ip_bytes',
   'resp_pkts',
   'resp_ip_bytes'],
  'categorical_columns': ['proto', 'service', 'conn_state', 'history'],
  'excluded_columns': ['ts'],
  'note': 'Timestamp is kept for splitting/metadata but is not used as a model feature.'},
 'node_feature_policy': {'mode': 'ip_stats',
  'ip_features': ['ipv4_octet_1',
   'ipv4_octet_2',
   'ipv4_octet_3',
   'ipv4_octet_4',
   'is_private',
   'is_global',
   'is_multicast',
   'is_loopback'],
  'train_only_stat_features': ['train_scaled_log1p_out_degree',
   'train_scaled_log1p_in_degree',
   'train_scaled_log1p_total_degree',
   'train_scaled_log1p_unique_out_peers',
   'train_scaled_log1p_unique_in_peers',
   

In [11]:
import numpy as np

arrays = np.load(STRICT_DIR / "graph_arrays.npz")
print(arrays["x"].shape)

print(arrays["x"][:3, -3:])

graph_metadata["node_feature_names"][-1]

(92413, 20)
[[10.712251    6.798932   31.04409   ]
 [-0.48732793 -0.48833153 -0.04162829]
 [-0.48732793 -0.48833153 -0.04162829]]


'train_scaled_log1p_avg_packet_size'

## Inspect strict results

The training commands used for the current saved results:

```powershell
python -m src.train_gat --arrays-path data\strict_time_balanced\graph_arrays.npz --model-dir models\strict_time_balanced --results-dir results\strict_time_balanced --epochs 50 --hidden-channels 64 --heads 4 --early-stopping --patience 8 --selection-metric val_tuned_f1 --message-passing-edges train
python -m src.train_baselines --arrays-path data\strict_time_balanced\graph_arrays.npz --model-dir models\strict_time_balanced --results-dir results\strict_time_balanced
```

In [6]:
comparison = pd.read_csv(STRICT_RESULTS / "model_comparison.csv")
comparison

,model,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc,selected_by
0,LogisticRegression,0.971689,0.998533,0.997207,0.999867,0.998535,0.997376,0.993593,validation_f1
1,MLP,0.999995,0.833600,0.997811,0.668667,0.800734,0.945100,0.948052,validation_f1
2,RandomForest,0.696039,0.513267,0.506723,0.999867,0.672586,0.028964,0.348523,validation_f1
3,DummyMostFrequent,0.000000,0.500000,0.500000,1.000000,0.666667,0.500000,0.500000,validation_f1


In [7]:
gat_metrics = json.loads((STRICT_RESULTS / "gat_metrics.json").read_text(encoding="utf-8"))
gat_metrics["selection_policy"], gat_metrics["final_metrics"]["test"]

({'best_model_selected_by': 'val_tuned_f1',
  'classification_threshold_selected_by': 'validation_f1_precision_recall_curve',
  'test_set_usage': 'reported_once_after_model_and_threshold_selection',
  'message_passing_edges': 'train',
  'selection_score': 0.8227346541084769},
 {'threshold': 0.994300365447998,
  'accuracy': 0.7098,
  'precision': 0.6973040752351097,
  'recall': 0.7414666666666667,
  'f1': 0.7187075928917609,
  'roc_auc': 0.6871447111111111,
  'pr_auc': 0.594454895924266,
  'confusion_matrix': {'tn': 5086, 'fp': 2414, 'fn': 1939, 'tp': 5561}})